# Modelo 2 — Clustering de Productos por Perfil de Volatilidad
**Proyecto:** food-price-forecasting-peru | Big Data Analytics UNMSM 2025-II

**Pregunta:** Existen grupos de alimentos con patrones de volatilidad similares?

**Algoritmos:** KMeans (K=2,3,4) vs Agglomerative Clustering (Ward)

**Flujo:** Cargar Plata → GroupBy producto → Limpiar matriz → Escalar → Clustering → MLflow → Zona Oro

In [ ]:
# CELDA 1 — Instalacion (ejecutar una vez por sesion)
#!pip install -q pyspark mlflow google-cloud-storage scikit-learn matplotlib scipy

In [ ]:
# CELDA 2 — Variables de configuracion
# Cambiar BUCKET_NAME y PROJECT_ID segun el entorno real
BUCKET_NAME = "food-price-peru-2025-01"
PROJECT_ID  = "tu-proyecto-gcp-id"   # <- Cambiar por el ID real de GCP

PATH_PLATA = f"gs://{BUCKET_NAME}/plata/precios_clima_integrado/precios_clima_parquet/"
PATH_ORO   = f"gs://{BUCKET_NAME}/oro/clustering_productos/"

print(f"Entrada : {PATH_PLATA}")
print(f"Salida  : {PATH_ORO}")

In [ ]:
# CELDA 3 — Autenticacion GCP e inicializacion de Spark
import os
from google.colab import auth
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

auth.authenticate_user()
!gcloud config set project {PROJECT_ID}

jar_name = "gcs-connector.jar"
jar_path = os.path.abspath(jar_name)
if not os.path.exists(jar_path):
    !wget -q https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.14/gcs-connector-hadoop3-2.2.14-shaded.jar -O {jar_name}

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {jar_path} pyspark-shell"

spark = (SparkSession.builder
    .appName("D08-Modelo2-Clustering")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.driver.extraClassPath", jar_path)
    .config("spark.executor.extraClassPath", jar_path)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado. Version:", spark.version)

In [ ]:
# CELDA 4 — Cargar data desde Zona Plata
# Contiene 10 productos x ~522 semanas = ~5,225 filas de precios+clima integrados
df_plata = spark.read.parquet(PATH_PLATA)

print("Schema:")
df_plata.printSchema()
print(f"Total filas: {df_plata.count()}")
print("Productos:")
df_plata.groupBy("producto").count().orderBy("producto").show(truncate=False)

In [ ]:
# CELDA 5 — GroupBy por producto: construir la matriz de clustering
# ─────────────────────────────────────────────────────────────────
# El KMeans NO entrena con las 5,225 filas semanales.
# Entrena con un perfil historico por producto: 10 filas, una por producto.
# Cada fila resume el comportamiento de ese producto en 2015-2024.
#
# Nota sobre dias_helada: todos los valores son 0 en la data actual.
# Se incluye en el groupby pero se descartara en la celda siguiente
# porque varianza=0 invalida el StandardScaler (division por cero).
# ─────────────────────────────────────────────────────────────────

df_matriz_spark = df_plata.groupBy("producto").agg(
    F.mean("precio_promedio").alias("precio_promedio_historico"),
    F.stddev("precio_promedio").alias("volatilidad"),
    F.max("precio_promedio").alias("precio_maximo_historico"),
    F.min("precio_promedio").alias("precio_minimo_historico"),
    F.mean("dias_lluvia_intensa").alias("exposicion_lluvia"),
    F.sum(F.when(F.col("dias_helada") > 0, 1).otherwise(0)).alias("semanas_con_helada")
)

# Solo 10 filas -> seguro pasar a pandas para sklearn
pdf_matriz = df_matriz_spark.toPandas()

print("MATRIZ DE CLUSTERING — 10 productos x 6 features")
print(pdf_matriz.to_string(index=False))
print("\nNulos por columna:")
print(pdf_matriz.isnull().sum())

In [ ]:
# CELDA 6 — Limpieza de la matriz
# ─────────────────────────────────────────────────────────────────
# Dos problemas detectados en la data real:
#
# 1. semanas_con_helada = 0 para todos los productos (varianza cero).
#    StandardScaler dividiria entre 0 -> NaN. Se descarta.
#
# 2. exposicion_lluvia = NaN para 4 productos procesados o de origen
#    animal (Fideos, Huevos, Leche, Pollo). No tienen estacion SENAMHI
#    asociada. Se imputa con 0 (sin exposicion a lluvia agricola).
# ─────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np

pdf_clean = pdf_matriz.copy()

# 1. Descartar columna con varianza cero
pdf_clean = pdf_clean.drop(columns=["semanas_con_helada"])
print("Descartada: semanas_con_helada (varianza=0 en todos los productos)")

# 2. Imputar NaN en exposicion_lluvia con 0
sin_lluvia = pdf_clean[pdf_clean["exposicion_lluvia"].isnull()]["producto"].tolist()
print(f"Imputados con exposicion_lluvia=0: {sin_lluvia}")
pdf_clean["exposicion_lluvia"] = pdf_clean["exposicion_lluvia"].fillna(0)

print(f"\nNulos tras limpieza: {pdf_clean.isnull().sum().sum()}")
print(pdf_clean.to_string(index=False))

In [ ]:
# CELDA 7 — Escalado con StandardScaler
# ─────────────────────────────────────────────────────────────────
# KMeans usa distancias euclidianas. Sin escalar, el ajo (precio 0-33)
# domina sobre la zanahoria (precio 0-2) solo por su magnitud.
# StandardScaler convierte cada variable a media=0 y std=1.
# ─────────────────────────────────────────────────────────────────

from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [
    "precio_promedio_historico",
    "volatilidad",
    "precio_maximo_historico",
    "precio_minimo_historico",
    "exposicion_lluvia"
]

productos  = pdf_clean["producto"].values
X          = pdf_clean[FEATURE_COLS].values
scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(X)

print("Features del clustering:", FEATURE_COLS)
df_scaled_check = pd.DataFrame(X_scaled, columns=FEATURE_COLS, index=productos)
print("\nMatriz escalada:")
print(df_scaled_check.round(3))

In [ ]:
# CELDA 8 — KMeans K=2, K=3, K=4 con MLflow
# ─────────────────────────────────────────────────────────────────
# Silhouette Score: mide compacidad y separacion de clusters.
# Rango -1 a 1. Meta del proyecto: > 0.40.
# Se registran parametros, metricas y artefacto del modelo en MLflow.
# ─────────────────────────────────────────────────────────────────

import mlflow
import mlflow.sklearn
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

mlflow.set_tracking_uri("file:///content/mlruns")
mlflow.set_experiment("D08_Modelo2_Clustering")

resultados_kmeans = []

for k in [2, 3, 4]:
    with mlflow.start_run(run_name=f"KMeans_K{k}"):
        modelo   = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels   = modelo.fit_predict(X_scaled)
        sil      = round(silhouette_score(X_scaled, labels), 4)
        inertia  = round(modelo.inertia_, 4)

        mlflow.log_param("algoritmo",    "KMeans")
        mlflow.log_param("k",            k)
        mlflow.log_param("n_init",       10)
        mlflow.log_param("random_state", 42)
        mlflow.log_metric("silhouette_score", sil)
        mlflow.log_metric("inertia",          inertia)
        mlflow.sklearn.log_model(modelo, f"modelo_kmeans_k{k}")

        resultados_kmeans.append({"nombre": f"KMeans K={k}", "K": k,
                                   "silhouette": sil, "inertia": inertia,
                                   "labels": labels, "modelo": modelo})
        print(f"KMeans K={k} -> Silhouette: {sil} | Inertia: {inertia}")

mejor_kmeans = max(resultados_kmeans, key=lambda x: x["silhouette"])
print(f"\nMejor KMeans: {mejor_kmeans['nombre']} -> Silhouette={mejor_kmeans['silhouette']}")

In [ ]:
# CELDA 9 — Agglomerative Clustering (Jerarquico) — algoritmo de comparacion
# ─────────────────────────────────────────────────────────────────
# Por que Agglomerative:
# KMeans requiere definir K y asume clusters esfericos.
# Agglomerative construye una jerarquia de fusiones desde abajo
# y no asume forma de cluster. Con 10 productos el dendrograma
# es perfectamente legible y muestra visualmente cuales productos
# son mas parecidos entre si, util para la presentacion final.
# ─────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

resultados_agg = []

for k in [2, 3, 4]:
    with mlflow.start_run(run_name=f"Agglomerative_K{k}"):
        modelo_agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels_agg = modelo_agg.fit_predict(X_scaled)
        sil_agg    = round(silhouette_score(X_scaled, labels_agg), 4)

        mlflow.log_param("algoritmo", "AgglomerativeClustering")
        mlflow.log_param("k",         k)
        mlflow.log_param("linkage",   "ward")
        mlflow.log_metric("silhouette_score", sil_agg)

        resultados_agg.append({"nombre": f"Agglomerative K={k}", "K": k,
                                "silhouette": sil_agg, "labels": labels_agg})
        print(f"Agglomerative K={k} -> Silhouette: {sil_agg}")

mejor_agg = max(resultados_agg, key=lambda x: x["silhouette"])
print(f"\nMejor Agglomerative: {mejor_agg['nombre']} -> Silhouette={mejor_agg['silhouette']}")

# Dendrograma — visualizacion de la jerarquia de fusiones
Z = linkage(X_scaled, method="ward")
plt.figure(figsize=(10, 5))
dendrogram(Z, labels=list(productos), leaf_rotation=45, leaf_font_size=9)
plt.title("Dendrograma — Clustering Jerarquico de Productos")
plt.xlabel("Producto")
plt.ylabel("Distancia de fusion (Ward)")
plt.tight_layout()
plt.savefig("dendrograma_productos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Dendrograma guardado.")

In [ ]:
# CELDA 10 — Comparativa y seleccion del mejor modelo
todos = resultados_kmeans + resultados_agg
ganador = max(todos, key=lambda x: x["silhouette"])

print("="*55)
print("COMPARATIVA — Silhouette Score")
print("="*55)
print(f"  Algoritmo                   K    Silhouette")
print("-"*55)
for r in todos:
    print(f"  {r['nombre']:<28} {r['K']}    {r['silhouette']}")

print(f"\nGANADOR: {ganador['nombre']} | Silhouette={ganador['silhouette']}")

# Tabla de asignaciones del ganador
df_resultado = pd.DataFrame({"producto": productos, "cluster": ganador["labels"]})
df_resultado = df_resultado.merge(pdf_clean[["producto"]+FEATURE_COLS], on="producto")

print("\nAsignacion de clusters:")
for cl in sorted(df_resultado["cluster"].unique()):
    prods = df_resultado[df_resultado["cluster"]==cl]["producto"].tolist()
    print(f"  Cluster {cl}: {prods}")

In [ ]:
# CELDA 11 — Visualizacion: precio promedio vs volatilidad por cluster
# Este es el grafico que va en la Vista 3 del dashboard.

import matplotlib.pyplot as plt

colores = ["#1A5FA8", "#0F7B6C", "#B07500", "#991F1F", "#4B0082"]
fig, ax = plt.subplots(figsize=(11, 6))

for cl in sorted(df_resultado["cluster"].unique()):
    mask = df_resultado["cluster"] == cl
    ax.scatter(
        df_resultado.loc[mask, "precio_promedio_historico"],
        df_resultado.loc[mask, "volatilidad"],
        color=colores[cl], s=220, label=f"Cluster {cl}", zorder=3
    )
    for _, row in df_resultado[mask].iterrows():
        ax.annotate(row["producto"],
            (row["precio_promedio_historico"], row["volatilidad"]),
            textcoords="offset points", xytext=(6, 4), fontsize=8)

ax.set_xlabel("Precio Promedio Historico (S/./kg)")
ax.set_ylabel("Volatilidad (Desviacion Estandar Historica)")
ax.set_title(f"Segmentacion de Productos por Perfil de Riesgo\n{ganador['nombre']}")
ax.legend(title="Cluster")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("clustering_productos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafico guardado como clustering_productos.png")

In [ ]:
# CELDA 12 — Guardar en Zona Oro (GCS) como Parquet
# ─────────────────────────────────────────────────────────────────
# La tabla Oro tiene 10 filas: una por producto con cluster asignado
# y sus features originales sin escalar para interpretabilidad.
# Esta tabla la consume el dashboard en la Vista 3.
# ─────────────────────────────────────────────────────────────────

df_resultado["modelo"]     = ganador["nombre"]
df_resultado["silhouette"] = ganador["silhouette"]

df_resultado_spark = spark.createDataFrame(df_resultado)
df_resultado_spark.write.mode("overwrite").parquet(PATH_ORO)
print(f"Guardado en Zona Oro: {PATH_ORO}")

# Verificar lectura
df_check = spark.read.parquet(PATH_ORO)
print(f"Filas verificadas: {df_check.count()}")
df_check.show(truncate=False)

# Registrar artefactos del ganador en MLflow
run_name_ganador = "GANADOR_" + ganador["nombre"].replace(" ", "_")
with mlflow.start_run(run_name=run_name_ganador):
    mlflow.log_param("modelo_ganador",    ganador["nombre"])
    mlflow.log_metric("silhouette_score", ganador["silhouette"])
    mlflow.log_artifact("clustering_productos.png")
    mlflow.log_artifact("dendrograma_productos.png")
print("Artefactos del ganador registrados en MLflow.")

In [ ]:
# CELDA 13 — Conclusion accionable del Modelo 2
# Copiar esta seccion al reporte final (Entregable 7, Seccion 4 Modelado)

print("="*60)
print("CONCLUSION ACCIONABLE — MODELO 2 CLUSTERING")
print("="*60)
print(f"Algoritmo ganador  : {ganador['nombre']}")
print(f"Silhouette Score   : {ganador['silhouette']} (meta: > 0.40)")
print()
print("Interpretacion:")
print("  El clustering segmenta los productos por nivel de riesgo")
print("  inflacionario. Productos con alta volatilidad y precio alto")
print("  como el Ajo requieren monitoreo mas frecuente y alertas")
print("  mas tempranas. Productos estables como la Zanahoria o el")
print("  Platano toleran ciclos de alerta mas largos.")
print()
print("Decision de politica publica habilitada:")
print("  El MIDAGRI puede priorizar recursos de monitoreo segun")
print("  el cluster: los productos de alta volatilidad justifican")
print("  revision semanal, los estables pueden revisarse mensualmente.")
print()
print("Comparativa registrada en MLflow:")
print("  KMeans K=2, K=3, K=4 | Agglomerative K=2, K=3, K=4")
print(f"Tabla en Zona Oro: {PATH_ORO}")